In [1]:
#importing the necessary packages
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
#Get the excel data
data=pd.read_csv("MSC_Agrigento_Data (1).csv")

In [3]:
data.head()

,VSL,IMO_No,VOY,REPORT_DATE_TIME,Report_Type,Status,LAT,LONG_,LATITUDE,LONGITUDE,...,FUEL_Boiler_Rsdl_ULS,FUEL_Boiler_Dstlt_VLS,FUEL_Boiler_Dstlt_ULS,FUEL_Boiler_Tnktnr_Dstlt_VLS,MANVRNG_MILES_BY_GPS,MANVRNG_MILES_BY_SPEED_LOG,Fuel_Type,Sulphur_Content_PC,Quantity,Port_Delivery
0,MSC AGRIGENTO,9618276.0,NL829R,2018-07-27 05:30:00,SAIL,IN PORT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
1,MSC AGRIGENTO,9618276.0,NL839R,2018-10-06 12:00:00,PORT,IN PORT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
2,MSC AGRIGENTO,9618276.0,NX708R,2017-03-03 12:00:00,NOON,AT SEA,0342S,08140W,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
3,MSC AGRIGENTO,9618276.0,FI124A,2021-08-18 09:48:00,COSP,DRIFTING,2655S,04834W,-26.916667,-48.566667,...,0.0,0.0,0.0,0.0,5.7,5.7,NaN,NaN,NaN,NaN
4,MSC AGRIGENTO,9618276.0,NX746R,2017-12-02 12:00:00,NOON,AT SEA,2328N,06323W,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN


In [4]:
df=data.copy() 

In [ ]:
df= df[df['report_date_time'] == 2024]

In [ ]:
df.dtypes #checking the variable types

In [ ]:
#Report time and date is not in correct format. So changing into correct format
df['REPORT_DATE_TIME'] = pd.to_datetime(df['REPORT_DATE_TIME'], format='%Y-%m-%d %H:%M:%S', errors='coerce') #changing the type

In [ ]:
#splitting data into date and time
df['Date']= df['REPORT_DATE_TIME'].dt.date
df['Time'] = df['REPORT_DATE_TIME'].dt.time

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
df.drop(columns=['REPORT_DATE_TIME'], inplace=True)

'''
Carbon Intensity Indicator (CII):
----> Measure of a ship’s pollution capacity. 
----> It compares a ship’s carbon emissions to the benefits it provides to society by moving goods through the sea
----> Bigger ships have higher emissions, but they also carry more cargo. Thus, the CII can give a fair evaluation of a 
----> ship’s air pollution potential irrespective of size and type of propulsion.

The CII rating has five main grades: A, B, C, D and E. The efficiency of the vessel decreases as we go from A to E. 

All vessels must aspire to be at least C-rated. 

Vessels graded D and E for three years must improve their score.

Ratings from A to E are classified as:

A – Major superior performance level
B – Minor superior performance level
C – Moderate performance level
D – Minor Inferior performance level
E – Inferior performance level

'''

# CII

In [ ]:
#Entering the datas into dictionary
ships = {
    'BULK': {'D1': 0.86, 'D2': 0.94, 'D3': 1.06, 'D4': 1.18,'A': 4745, 'C': 0.622},
    'TANKER': {'D1': 0.82, 'D2': 0.93, 'D3': 1.08, 'D4': 1.28, 'A': 5247, 'C': 0.610},
    'CONTAINER': {'D1': 0.83, 'D2': 0.94, 'D3': 1.07, 'D4': 1.19, 'A': 1984, 'C': 0.489},
    'COMBINATION': {'D1': 0.87, 'D2': 0.96, 'D3': 1.06, 'D4': 1.14, 'A': 40853, 'C': 0.812},
    'GENERAL CARGO SHIP': {'D1': 0.83, 'D2': 0.94, 'D3': 1.06, 'D4': 1.19, 'A': 10952, 'C': 0.637},
}

def calculate_cii_rating():
    ship_type = input("Enter the vessel type: ").strip().upper()   #vessel type from user
    dwt = float(input("Enter the deadweight (DWT): "))             #deadweight of the vessel
    cii_year = int(input("Enter a year (2014-2022): "))
    Res_df = df[df['Date'].dt.year == cii_year]
    print("*****************************************************************")
    if ship_type not in ships:                                     #checking if the vessel typpe is there or not
        print("Invalid ship type")
        return

    ship_info = ships[ship_type]                                   #getting a and c parameter from dictionary for CII-REF
    a = ship_info['A']
    c = ship_info['C']
    #a=1984
    #c=0.489

    # CII_REF
    CII_REF = a * (dwt ** (-c))
    print("CII_REF IS : {:.2f}".format(CII_REF))
    print("*****************************************************************")

    years = {                                                     #Corresponding year and their reduction %(Z)
        2019: 0,
        2020: 1,
        2021: 2,
        2022: 3,
        2023: 5,
        2024: 7,
        2025: 9,
        2026: 11
    }

    if cii_year in years:
        Z = years[cii_year]                                            
        print(f"Z value for year {cii_year} is {Z}")
    else:
        print("No Z value for the entered year")
        return

    #CII_REQ
    CII_REQ = CII_REF * ((100 - Z) / 100)
    print("CII_REQUIRED IS : {:.2f}".format(CII_REQ))
    print("*****************************************************************")

    #ATTAINED CII
    #Total HS
    Res_df['Total_Calculated_HS'] =  Res_df[['FUEL_M_E_Rsdl_HS' , 'FUEL_Aux_Rsdl_HS' , 'FUEL_Boiler_Rsdl_HS']].sum(axis=1)
    #Total LS
    Res_df['Total_Calculated_LS'] =  Res_df[['FUEL_M_E_Rsdl_VLS' , 'FUEL_Aux_Rsdl_VLS' , 'FUEL_Boiler_Rsdl_VLS' ,'FUEL_M_E_Rsdl_ULS' ,  'FUEL_Aux_Rsdl_ULS' , 'FUEL_Boiler_Rsdl_ULS']].sum(axis=1)
    #Total ULS
    Res_df['Total_Calculated_ULS'] =  Res_df[['FUEL_M_E_Dstlt_VLS' , 'FUEL_Aux_Dstlt_VLS' , 'FUEL_Boiler_Dstlt_VLS' ,'FUEL_M_E_Dstlt_ULS' ,'FUEL_Aux_Dstlt_ULS' , 'FUEL_Boiler_Dstlt_ULS' ,'FUEL_M_E_Tnktnr_Dstlt_VLS' , 'FUEL_Aux_Tnktnr_Dstlt_VLS' ,'FUEL_Boiler_Tnktnr_Dstlt_VLS']].sum(axis=1)
    #co2 factor
    Res_df['CO2_HFC']=Res_df['Total_Calculated_HS']*3.114
    Res_df['CO2_LFC']=Res_df['Total_Calculated_LS']*3.151
    Res_df['CO2_UFC']=Res_df['Total_Calculated_ULS']*3.206
    
    Res_df['Mass_FC']=Res_df[['CO2_HFC','CO2_LFC','CO2_UFC']].sum(axis=1)
    
    #Transport for work
    dist = Res_df['MILES_BY_GPS'].sum()
    TAFC = Res_df['Mass_FC'].sum() * 10 ** 6
    ATT_CII = TAFC / (dist * dwt)
    print("ATTAINED CII IS : {:.3f}".format(ATT_CII))
    print("*****************************************************************")
   

    #CII RATIO
    CII_RATIO = ATT_CII / CII_REQ
    print("CII_RATIO IS : {:.2f}".format(CII_RATIO))
    print("*****************************************************************")

    #RATING
    if CII_RATIO <= ship_info['D1']:
        rating = 'A RATING'
    elif ship_info['D1'] < CII_RATIO <= ship_info['D2']:
        rating = 'B RATING'
    elif ship_info['D2'] < CII_RATIO <= ship_info['D3']:
        rating = 'C RATING'
    elif ship_info['D3'] < CII_RATIO <= ship_info['D4']:
        rating = 'D RATING'
    else:
        rating = 'E RATING'

    print(f"The CII rating MSC AGRIGENTO in year {cii_year} is {rating}")


calculate_cii_rating()



In [ ]:
#110652.40 from dbd (old)
#127951.80 from analytics 

In [ ]:
import plotly.graph_objects as go

years = [2019, 2020, 2021, 2022, 2023]
attained_cii = [1.96, 7.74, 7.52, 8.03, 6.80]
required_cii = [6.31, 6.25, 6.12, 6.19, 6.00]


cii_ratio = [att / req for att, req in zip(attained_cii, required_cii)]

d1_range = (0, 0.83)
d2_range = (0.83, 0.94)
d3_range = (0.94, 1.07)
d4_range = (1.07, 1.19)
d5_range = (1.19, float('inf'))

ratings = []
for ratio in cii_ratio:
    if ratio <= d1_range[1]:
        ratings.append('A RATING')
    elif d2_range[0] < ratio <= d2_range[1]:
        ratings.append('B RATING')
    elif d3_range[0] < ratio <= d3_range[1]:
        ratings.append('C RATING')
    elif d4_range[0] < ratio <= d4_range[1]:
        ratings.append('D RATING')
    else:
        ratings.append('E RATING')

colors = {
    'A RATING': 'green',
    'B RATING': 'lightgreen',
    'C RATING': 'yellow',
    'D RATING': 'orange',
    'E RATING': 'red'
}

cii_ratio_colors = [colors.get(rate, 'gray') for rate in ratings]

fig = go.Figure()

fig.add_trace(go.Bar(x=years, y=attained_cii, marker=dict(color=cii_ratio_colors), name='Attained CII'))

fig.add_trace(go.Scatter(x=years, y=required_cii, mode='lines+markers', name='Required CII', line=dict(color='cyan')))

fig.update_layout(
    title='Attained CII and CII Ratings Over Time',
    xaxis_title='Year',
    yaxis_title='CII Value',
    legend=dict(x=0, y=1)
)

fig.show()



In [ ]:
import plotly.graph_objects as go

years = ['2019', '2020', '2021', '2022', '2023']  
attained_cii = [1.96, 7.74, 7.52, 8.03, 6.80]
required_cii = [6.31, 6.25, 6.12, 6.19, 6.00]

fig = go.Figure()

fig.add_trace(go.Scatter(x=years, y=attained_cii, mode='lines+markers', name='Attained CII',
            line=dict(color='blue', width=3), marker=dict(color='blue', size=10, line=dict(color='white', width=2))))

fig.add_trace(go.Scatter(x=years, y=required_cii, mode='lines+markers', name='Required CII',
                         line=dict(color='cyan', width=2, dash='dash'), marker=dict(color='cyan', size=10, line=dict(color='white', width=2))))

fig.update_layout(
    title='CII',
    xaxis_title='Year',
    yaxis_title='CII Value',
    xaxis=dict(categoryorder='array', categoryarray=years),  
    plot_bgcolor='rgba(245,245,245,0.7)',  
    paper_bgcolor='white',  
    font=dict(family='Arial', size=12), 
    showlegend=True,
    legend=dict(x=0.02, y=0.95)  
)



fig.show()


In [ ]:
import plotly.graph_objects as go

years = [2019, 2020, 2021, 2022, 2023]
attained_cii = [1.96, 7.74, 7.52, 8.03, 6.80]
required_cii = [6.31, 6.25, 6.12, 6.19, 6.00]

fig = go.Figure()

# Create a shaded region between Attained CII and Required CII lines
fig.add_trace(go.Scatter(x=years, y=attained_cii, mode='lines+markers', name='Attained CII', line=dict(color='blue'),fill='tozeroy'))
fig.add_trace(go.Scatter(x=years, y=required_cii, mode='lines+markers', name='Required CII', line=dict(color='cyan', width=2, dash='dash'),fill='tozeroy'))

fig.update_layout(
    title=' CII ',
    xaxis_title='Year',
    yaxis_title='CII Value',
    plot_bgcolor='rgba(225,255,150,0.7)',  
    paper_bgcolor='white',
    legend=dict(x=0, y=1),
    xaxis=dict(type='category'), 
)

fig.show()